In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

REPO_URL = "https://github.com/Rheacoutinho/humanoid-perception-3d-reconstruction"
REPO_DIR = "/content/humanoid-perception-3d-reconstruction"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repo already cloned")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

Repo already cloned
Working directory: /content/humanoid-perception-3d-reconstruction


In [4]:
import os

MAST3R_DIR = "/content/mast3r"

if not os.path.exists(MAST3R_DIR):
    os.system("git clone --recursive https://github.com/naver/mast3r.git /content/mast3r")
    os.chdir("/content/mast3r")
    os.system("pip install -e '.[demo]' --quiet")
    os.chdir(REPO_DIR)
    print("MASt3R installed")
else:
    print("MASt3R already installed")

import sys
sys.path.insert(0, "/content/mast3r")
sys.path.insert(0, "/content/mast3r/dust3r")
print("MASt3R added to path")

MASt3R installed
MASt3R added to path


In [5]:
SAM2_DIR = "/content/sam2"

if not os.path.exists(SAM2_DIR):
    os.system("git clone https://github.com/facebookresearch/sam2.git /content/sam2")
    os.chdir("/content/sam2")
    os.system("pip install -e . --quiet")
    os.chdir(REPO_DIR)
    print("SAM2 installed")
else:
    print("SAM2 already installed")

sys.path.insert(0, "/content/sam2")
print("SAM2 added to path")

SAM2 installed
SAM2 added to path


In [6]:
os.system("pip install open3d transformers gradio opencv-python scipy matplotlib tqdm --quiet")
print("All packages installed")

All packages installed


In [7]:
import os

CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# MASt3R checkpoint
mast3r_ckpt = f"{CHECKPOINT_DIR}/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
if not os.path.exists(mast3r_ckpt):
    print("Downloading MASt3R checkpoint (~1.4GB)...")
    os.system(f"wget -q --show-progress -O {mast3r_ckpt} "
              "https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth")
    print("MASt3R checkpoint downloaded")
else:
    print("MASt3R checkpoint already exists")

# SAM2 checkpoint
sam2_ckpt = f"{CHECKPOINT_DIR}/sam2.1_hiera_large.pt"
if not os.path.exists(sam2_ckpt):
    print("Downloading SAM2 checkpoint (~900MB)...")
    os.system(f"wget -q --show-progress -O {sam2_ckpt} "
              "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt")
    print("SAM2 checkpoint downloaded")
else:
    print("SAM2 checkpoint already exists")

print("\nAll checkpoints ready.")

MASt3R checkpoint downloaded
SAM2 checkpoint downloaded

All checkpoints ready.


In [8]:
from transformers import pipeline
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# This downloads the model on first run (~400MB) and caches it
depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=0 if device == "cuda" else -1
)
print("Depth-Anything V2 loaded successfully")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Depth-Anything V2 loaded successfully


In [9]:
import cv2
import open3d as o3d
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import gradio as gr

# Quick smoke test
print("opencv:", cv2.__version__)
print("open3d:", o3d.__version__)
print("numpy:", np.__version__)

# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

print("\nAll imports successful. Environment is ready.")

opencv: 4.13.0
open3d: 0.19.0
numpy: 2.0.2
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

All imports successful. Environment is ready.
